## Sam Packer Exploratory Data Analysis

Import the necessary dependencies for us to work with

In [73]:
import pandas as pd

Load the relevant datasets. For this example, we'll look at steps taken and calories burned by people during COVID-19 symptoms and while healthy. We're interested in seeing how healthy people were in general and if they were able to still get exercise during their symptoms. It's likely the more severe symptoms, the more rest they needed and didn't take as many steps as usual.

In [74]:
participants_df = pd.read_csv("../data/participants.csv")
wearables_df = pd.read_csv("../data/wearables.csv")
surveys_df = pd.read_csv("../data/surveys.csv")
print("Data structure:")
print("Participants:")
print(participants_df.shape)
print(participants_df.dtypes)
print("--------------------")
print("Wearables:")
print(wearables_df.shape)
print(wearables_df.dtypes)
print("--------------------")
print("Surveys:")
print(surveys_df.shape)
print(surveys_df.dtypes)
print("--------------------")

Data structure:
Participants:
(185, 8)
user_code          object
gender             object
age_range          object
city               object
country            object
height            float64
weight            float64
symptoms_onset     object
dtype: object
--------------------
Wearables:
(3098, 18)
user_code                           object
day                                 object
resting_pulse                      float64
pulse_average                      float64
pulse_min                          float64
pulse_max                          float64
average_spo2_value                 float64
body_temperature_avg               float64
stand_hours_total                  float64
steps_count                        float64
distance                           float64
steps_speed                        float64
total_number_of_flights_climbed    float64
active_calories_burned             float64
basal_calories_burned              float64
total_calories_burned              float64
average_

We need to normalize the data. The survey data has multiple rows per each response on a given day. We want to turn that into only one row with the responses being in a column. This way, we can have our data cleanly merge into the wearables data.

Our plan is to drop the `text` value. That is simply an English explanation of the symptoms and won't really help with data analysis. It's also important to know that if the person did not have symptoms, the value is NaN. We should fill those in with a 0.0 instead.

We're also only really interested in COVID-19 symptoms. For now, we can clean up the dataset to remove a lot of the data that isn't related to COVID-19 symptoms. Keeping data about their health lifestyle may be important in the future for making predictions.

In [79]:
# Note: ChatGPT helped with the pivot table and the idea to widen the columns
pivot = surveys_df.pivot_table(index=["user_code", "created_at"], columns=["scale"], values=["value"], aggfunc="first")
pivot.columns = [f"{scale}_{attr}" for attr, scale in pivot.columns]
pivot = pivot.reset_index()

value_cols = [cl for cl in pivot.columns if cl.endswith("_value")]
pivot[value_cols] = pivot[value_cols].fillna(0.0)
print(pivot)

      user_code  created_at  S_CORONA_value  S_COVID_BLUISH_value  \
0    01bad5a519  2020-04-23             2.0                   1.0   
1    01bad5a519  2020-04-25             0.0                   1.0   
2    01bad5a519  2020-04-27             0.0                   1.0   
3    01bad5a519  2020-04-29             0.0                   1.0   
4    01bad5a519  2020-05-03             0.0                   1.0   
..          ...         ...             ...                   ...   
320  fde84801d8  2020-05-10             0.0                   1.0   
321  fde84801d8  2020-05-14             0.0                   1.0   
322  fde84801d8  2020-05-16             0.0                   1.0   
323  fe6c1b1349  2020-05-07             1.0                   0.0   
324  fe6c1b1349  2020-05-12             0.0                   1.0   

     S_COVID_BREATH_value  S_COVID_CONFUSION_value  S_COVID_COUGH_value  \
0                     3.0                      1.0                  2.0   
1                    

We can now merge our data in. We'll rename the `created_at` row to `day` and merge them both on the `user_code` and `day` DataFrame.

In [80]:
pivot.rename(columns={"created_at": "day"}, inplace=True)
pivot['day'] = pd.to_datetime(pivot['day'])
wearables_df['day'] = pd.to_datetime(sleep_df['day'])

merged_data = pd.merge(pivot, wearables_df, how="left", on=['user_code', 'day'])
print(merged_data.dtypes)

user_code                               object
day                             datetime64[ns]
S_CORONA_value                         float64
S_COVID_BLUISH_value                   float64
S_COVID_BREATH_value                   float64
                                     ...      
active_calories_burned                 float64
basal_calories_burned                  float64
total_calories_burned                  float64
average_headphone_exposure             float64
average_environment_exposure           float64
Length: 76, dtype: object


We now have our merged data! Let's save it to a CSV so we can now take a deep dive at it.

In [81]:
merged_data.to_csv("merged_data.csv")

In the future, I would actually implement logic to analyze these columns and compare them to the steps and potentially plot them out.